# BiLSTM Baseline -- WELFake Fake News Detection

This notebook trains a Bidirectional LSTM neural baseline
on the WELFake dataset. The BiLSTM captures sequential
context and semantic patterns that TF-IDF bag-of-words
cannot represent.

The best classical model (LinearSVC) achieved 97.22% F1 Macro.
The BiLSTM baseline target is to match or exceed this.

Pipeline:
1. Build vocabulary and tokenizer from training set only
2. Encode text as padded integer sequences
3. Run systematic hyperparameter experiments
4. Train final BiLSTM with optimal parameters
5. Evaluate on held-out test set
6. Generate training curves and evaluation plots

## Section 1 -- Setup and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import f1_score as sk_f1
import re
import joblib

BASE          = '/content/drive/MyDrive/MSU Semester 4/Applied ML/Project'
PROCESSED_DIR = BASE + '/processed/'
SRC_DIR       = BASE + '/src/'
MODELS_DIR    = BASE + '/models/'
FIGURES_DIR   = BASE + '/figures/results/'
RESULTS_DIR   = BASE + '/results/'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sys.path.append(SRC_DIR)

from models  import build_bilstm, train_model, save_pytorch_model, get_device
from evaluate import compute_metrics, build_results_table, save_results
from utils   import set_seed, print_section

set_seed(42)
device = get_device()
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f'Device         : {device}')
print(f'PyTorch version: {torch.__version__}')
gpu_name = os.popen(
    'nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print(f'GPU            : {gpu_name}')

## Section 2 -- Load Processed Data

The three splits saved during preprocessing are loaded here.
Only the content and label columns are needed for this notebook.
Class balance is printed to confirm stratification held correctly.

In [ ]:
print_section('Loading Processed Data')

train_df = pd.read_csv(PROCESSED_DIR + 'train_clean.csv')
val_df   = pd.read_csv(PROCESSED_DIR + 'val_clean.csv')
test_df  = pd.read_csv(PROCESSED_DIR + 'test_clean.csv')

X_train_text = train_df['content'].fillna('').tolist()
X_val_text   = val_df['content'].fillna('').tolist()
X_test_text  = test_df['content'].fillna('').tolist()

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print(f'Train : {len(X_train_text):,} articles')
print(f'Val   : {len(X_val_text):,} articles')
print(f'Test  : {len(X_test_text):,} articles')
print()
print('Class balance -- Train:')
print(pd.Series(y_train).value_counts(normalize=True).round(4))

## Section 3 -- Build Tokenizer

A simple whitespace tokenizer is built from the training set only.
Vocabulary is capped at 30,000 tokens. Index 0 is reserved for
padding and index 1 for unknown tokens not seen during training.
Fitting on training data only prevents vocabulary leakage from
validation or test sets.

In [ ]:
print_section('Building Tokenizer')

VOCAB_SIZE = 30000
MAX_LEN    = 300

def tokenize(text):
    # Lowercase and remove punctuation except apostrophes
    return re.sub(r"[^\w\s']", '', text.lower()).split()

# Count word frequencies in training set only
counter = Counter()
for text in X_train_text:
    counter.update(tokenize(text))

print(f'Total unique tokens in training set: {len(counter):,}')

# Build vocab: index 0 = PAD, index 1 = UNK
vocab    = ['<PAD>', '<UNK>'] + [
    w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}
ACTUAL_VOCAB_SIZE = len(word2idx)

print(f'Vocabulary size (capped) : {ACTUAL_VOCAB_SIZE:,}')
print(f'Max sequence length      : {MAX_LEN}')

joblib.dump(word2idx, MODELS_DIR + 'tokenizer.joblib')
print('Tokenizer saved to models/tokenizer.joblib')

## Section 4 -- Encode and Pad Sequences

Each article is converted to a fixed-length integer sequence.
Sequences longer than MAX_LEN are truncated from the right.
Sequences shorter than MAX_LEN are zero-padded on the right.

In [ ]:
def encode_and_pad(texts, word2idx, max_len):
    """
    Convert list of text strings to padded integer sequences.
    Unknown tokens map to index 1. Padding token is index 0.
    """
    sequences = []
    for text in texts:
        tokens = tokenize(text)
        ids    = [word2idx.get(t, 1) for t in tokens]
        ids    = ids[:max_len]
        ids    = ids + [0] * (max_len - len(ids))
        sequences.append(ids)
    return np.array(sequences, dtype=np.int64)

print('Encoding sequences...')
X_train_seq = encode_and_pad(X_train_text, word2idx, MAX_LEN)
X_val_seq   = encode_and_pad(X_val_text,   word2idx, MAX_LEN)
X_test_seq  = encode_and_pad(X_test_text,  word2idx, MAX_LEN)

print(f'Train shape : {X_train_seq.shape}')
print(f'Val shape   : {X_val_seq.shape}')
print(f'Test shape  : {X_test_seq.shape}')

## Section 5 -- Create PyTorch Datasets and DataLoaders

A custom PyTorch Dataset wraps sequences and labels.
Training loader shuffles data each epoch to reduce overfitting.
Validation and test loaders do not shuffle.

In [ ]:
class TextDataset(Dataset):
    """
    PyTorch Dataset for padded integer text sequences.
    Returns (sequence_tensor, label_tensor) pairs.
    """
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

BATCH_SIZE = 64

train_loader = DataLoader(
    TextDataset(X_train_seq, y_train),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TextDataset(X_val_seq, y_val),
    batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    TextDataset(X_test_seq, y_test),
    batch_size=BATCH_SIZE, shuffle=False
)

print(f'Train batches : {len(train_loader):,}')
print(f'Val batches   : {len(val_loader):,}')
print(f'Test batches  : {len(test_loader):,}')

## Section 6 -- Hyperparameter Experiments

Three experiments systematically identify optimal hyperparameters.
Each experiment trains for 5 epochs maximum with early stopping
patience of 2. All experiments use the training and validation
sets only. The test set is not used at any point during tuning.

A reusable quick_train function handles each experiment run
to avoid code duplication.

In [ ]:
def quick_train(lr=0.001, lstm_units=128,
                embedding_dim=128, dropout=0.3,
                epochs=5, patience=2):
    """
    Short training run to evaluate a hyperparameter configuration.
    Returns best validation F1 achieved across all epochs.

    Parameters
    ----------
    lr : float
        Learning rate for Adam optimizer.
    lstm_units : int
        Number of LSTM units per direction.
    embedding_dim : int
        Word embedding dimension.
    dropout : float
        Dropout rate applied after dense layer.
    epochs : int
        Maximum epochs to train.
    patience : int
        Early stopping patience on val F1.

    Returns
    -------
    float
        Best validation F1 macro score.
    """
    model = build_bilstm(
        vocab_size=ACTUAL_VOCAB_SIZE,
        embedding_dim=embedding_dim,
        max_len=MAX_LEN,
        lstm_units=lstm_units,
        dropout=dropout
    ).to(device)

    optimizer   = torch.optim.Adam(model.parameters(), lr=lr)
    criterion   = nn.BCELoss()
    best_val_f1 = 0.0
    no_improve  = 0

    for epoch in range(epochs):
        # Training pass
        model.train()
        for seqs, labels in train_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            optimizer.zero_grad()
            preds = model(seqs).squeeze()
            loss  = criterion(preds, labels)
            loss.backward()
            optimizer.step()

        # Validation pass
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for seqs, labels in val_loader:
                seqs  = seqs.to(device)
                preds = model(seqs).squeeze().cpu().numpy()
                all_preds.extend((preds > 0.5).astype(int))
                all_labels.extend(labels.numpy().astype(int))

        val_f1 = sk_f1(all_labels, all_preds, average='macro')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve  = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    # Free GPU memory after each run
    del model
    torch.cuda.empty_cache()
    return round(best_val_f1, 4)

print('quick_train helper function defined.')

### Experiment A -- Learning Rate

Learning rate has the largest impact on neural network training.
Four values are tested from 0.01 (aggressive) down to 0.0001
(conservative). LSTM networks typically perform best in the
range 0.001 to 0.0001. Including 0.01 empirically confirms
whether the aggressive rate causes instability on this dataset.

In [ ]:
print_section('Experiment A: Learning Rate')

lr_candidates = [0.01, 0.001, 0.0005, 0.0001]
lr_results    = []

for lr in lr_candidates:
    print(f'Testing lr={lr}...')
    start   = time.time()
    val_f1  = quick_train(lr=lr)
    elapsed = round(time.time() - start, 1)
    lr_results.append({
        'lr': lr, 'val_f1': val_f1,
        'time_seconds': elapsed
    })
    print(f'  val_f1={val_f1:.4f}  time={elapsed}s')

lr_df      = pd.DataFrame(lr_results)
OPTIMAL_LR = float(lr_df.loc[lr_df['val_f1'].idxmax(), 'lr'])
print(f'\nExperiment A Results:')
print(lr_df.to_string(index=False))
print(f'\nOptimal learning rate: {OPTIMAL_LR}')

### Experiment B -- LSTM Units

The number of LSTM units controls model capacity. More units
capture richer sequential patterns but increase training time
and risk of overfitting. Three values from 64 to 256 are tested
using the optimal learning rate from Experiment A.

In [ ]:
print_section('Experiment B: LSTM Units')

lstm_candidates = [64, 128, 256]
lstm_results    = []

for units in lstm_candidates:
    print(f'Testing lstm_units={units}...')
    start   = time.time()
    val_f1  = quick_train(lr=OPTIMAL_LR, lstm_units=units)
    elapsed = round(time.time() - start, 1)
    lstm_results.append({
        'lstm_units': units,
        'val_f1':     val_f1,
        'time_seconds': elapsed
    })
    print(f'  val_f1={val_f1:.4f}  time={elapsed}s')

lstm_df      = pd.DataFrame(lstm_results)
OPTIMAL_LSTM = int(lstm_df.loc[
    lstm_df['val_f1'].idxmax(), 'lstm_units'])
print(f'\nExperiment B Results:')
print(lstm_df.to_string(index=False))
print(f'\nOptimal lstm_units: {OPTIMAL_LSTM}')

### Experiment C -- Dropout Rate

Dropout randomly deactivates neurons during training to reduce
overfitting. Too little dropout risks memorizing training data.
Too much dropout loses important signal. Three values are tested
using optimal lr and lstm_units from previous experiments.

In [ ]:
print_section('Experiment C: Dropout Rate')

dropout_candidates = [0.2, 0.3, 0.5]
dropout_results    = []

for dropout in dropout_candidates:
    print(f'Testing dropout={dropout}...')
    start   = time.time()
    val_f1  = quick_train(
        lr=OPTIMAL_LR,
        lstm_units=OPTIMAL_LSTM,
        dropout=dropout
    )
    elapsed = round(time.time() - start, 1)
    dropout_results.append({
        'dropout':      dropout,
        'val_f1':       val_f1,
        'time_seconds': elapsed
    })
    print(f'  val_f1={val_f1:.4f}  time={elapsed}s')

dropout_df      = pd.DataFrame(dropout_results)
OPTIMAL_DROPOUT = float(dropout_df.loc[
    dropout_df['val_f1'].idxmax(), 'dropout'])
print(f'\nExperiment C Results:')
print(dropout_df.to_string(index=False))
print(f'\nOptimal dropout: {OPTIMAL_DROPOUT}')

### Hyperparameter Experiment Summary

In [ ]:
print_section('Optimal Hyperparameter Configuration')
print(f'Learning rate : {OPTIMAL_LR}')
print(f'LSTM units    : {OPTIMAL_LSTM}')
print(f'Dropout       : {OPTIMAL_DROPOUT}')
print(f'Embedding dim : 128 (fixed)')
print(f'Batch size    : {BATCH_SIZE} (fixed)')
print(f'Max seq len   : {MAX_LEN} (fixed)')

## Section 7 -- Build Final BiLSTM Model

The final model is built using the optimal hyperparameters
identified across all three experiments. Architecture and
total parameter count are printed for the report.

In [ ]:
print_section('Building Final BiLSTM Model')

model = build_bilstm(
    vocab_size=ACTUAL_VOCAB_SIZE,
    embedding_dim=128,
    max_len=MAX_LEN,
    lstm_units=OPTIMAL_LSTM,
    dropout=OPTIMAL_DROPOUT
).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

## Section 8 -- Train Final Model

The final model is trained using optimal hyperparameters.
Early stopping monitors validation loss with patience of 3.
Best model weights are saved after each improvement and
restored at the end of training.

In [ ]:
print_section('Training Final BiLSTM Model')
print(f'Learning rate : {OPTIMAL_LR}')
print(f'LSTM units    : {OPTIMAL_LSTM}')
print(f'Dropout       : {OPTIMAL_DROPOUT}')
print(f'Max epochs    : 10')
print(f'Early stop    : patience=3')
print()

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,
    lr=OPTIMAL_LR,
    patience=3,
    device=str(device)
)

save_pytorch_model(model, MODELS_DIR + 'bilstm_model.pt')
print('Model saved to models/bilstm_model.pt')

## Section 9 -- Evaluate on Test Set

Final evaluation uses the held-out test set which has not
been used during any tuning stage. Results are compared
against the classical baseline LinearSVC at 97.20% F1.

In [ ]:
print_section('Evaluation on Test Set')

model.eval()
all_preds  = []
all_scores = []
all_labels = []

with torch.no_grad():
    for seqs, labels in test_loader:
        seqs   = seqs.to(device)
        scores = model(seqs).squeeze().cpu().numpy()
        preds  = (scores > 0.5).astype(int)
        all_scores.extend(scores.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().astype(int).tolist())

bilstm_metrics = compute_metrics(
    all_labels, all_preds, all_scores, 'BiLSTM'
)

print(f"Accuracy : {bilstm_metrics['Accuracy']:.4f}")
print(f"F1 Macro : {bilstm_metrics['F1_Macro']:.4f}")
print(f"ROC-AUC  : {bilstm_metrics['ROC_AUC']:.4f}")

results_df = build_results_table([bilstm_metrics])
save_results(results_df, RESULTS_DIR + 'bilstm_results.csv')

diff = bilstm_metrics['F1_Macro'] - 0.9720
print()
print('Comparison to best classical model:')
print(f'  LinearSVC F1 Macro : 0.9720')
print(f"  BiLSTM    F1 Macro : {bilstm_metrics['F1_Macro']:.4f}")
print(f'  Difference         : {diff:+.4f}')

## Section 10 -- Visualizations

Four plots are generated to support the results and methodology
sections of the written report and Streamlit application.

### Plot 15 -- Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'],
         color='#E76F51', linewidth=2,
         marker='o', label='Train loss')
ax1.plot(epochs_range, history['val_loss'],
         color='#2A9D8F', linewidth=2,
         marker='s', label='Val loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=13)
ax1.legend()

ax2.plot(epochs_range, history['train_acc'],
         color='#E76F51', linewidth=2,
         marker='o', label='Train accuracy')
ax2.plot(epochs_range, history['val_acc'],
         color='#2A9D8F', linewidth=2,
         marker='s', label='Val accuracy')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=13)
ax2.legend()

fig.suptitle('BiLSTM Training Curves',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '15_bilstm_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Plot 16 -- ROC Curve and Confusion Matrix

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(all_labels, all_scores)
auc_val     = roc_auc_score(all_labels, all_scores)

ax1.plot(fpr, tpr, color='#2A9D8F', linewidth=2,
         label=f'BiLSTM (AUC = {auc_val:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1,
         alpha=0.5, label='Random classifier')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve -- BiLSTM', fontsize=13)
ax1.legend()

cm     = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

sns.heatmap(cm_pct, annot=True, fmt='.1f',
            cmap='YlOrRd',
            xticklabels=['Fake (0)', 'Real (1)'],
            yticklabels=['Fake (0)', 'Real (1)'],
            ax=ax2, cbar=False)
for i in range(2):
    for j in range(2):
        ax2.text(j + 0.5, i + 0.72,
                 f'n={cm[i, j]:,}',
                 ha='center', va='center',
                 fontsize=9, color='gray')
ax2.set_title('Confusion Matrix -- BiLSTM', fontsize=13)
ax2.set_ylabel('True Label', fontsize=11)
ax2.set_xlabel('Predicted Label', fontsize=11)

fig.suptitle('BiLSTM Evaluation -- Test Set',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '16_bilstm_evaluation.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Plot 17 -- Hyperparameter Experiment Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Experiment A -- Learning Rate
axes[0].bar(
    [str(r['lr']) for r in lr_results],
    [r['val_f1'] for r in lr_results],
    color=['#2A9D8F' if r['lr'] == OPTIMAL_LR
           else '#E9C46A' for r in lr_results],
    alpha=0.85
)
for i, r in enumerate(lr_results):
    axes[0].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[0].set_xlabel('Learning Rate', fontsize=11)
axes[0].set_ylabel('Val F1 Macro', fontsize=11)
axes[0].set_title('Experiment A: Learning Rate', fontsize=12)
axes[0].set_ylim(
    min(r['val_f1'] for r in lr_results) - 0.02, 1.0)

# Experiment B -- LSTM Units
axes[1].bar(
    [str(r['lstm_units']) for r in lstm_results],
    [r['val_f1'] for r in lstm_results],
    color=['#2A9D8F' if r['lstm_units'] == OPTIMAL_LSTM
           else '#E9C46A' for r in lstm_results],
    alpha=0.85
)
for i, r in enumerate(lstm_results):
    axes[1].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[1].set_xlabel('LSTM Units', fontsize=11)
axes[1].set_ylabel('Val F1 Macro', fontsize=11)
axes[1].set_title('Experiment B: LSTM Units', fontsize=12)
axes[1].set_ylim(
    min(r['val_f1'] for r in lstm_results) - 0.02, 1.0)

# Experiment C -- Dropout
axes[2].bar(
    [str(r['dropout']) for r in dropout_results],
    [r['val_f1'] for r in dropout_results],
    color=['#2A9D8F' if r['dropout'] == OPTIMAL_DROPOUT
           else '#E9C46A' for r in dropout_results],
    alpha=0.85
)
for i, r in enumerate(dropout_results):
    axes[2].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[2].set_xlabel('Dropout Rate', fontsize=11)
axes[2].set_ylabel('Val F1 Macro', fontsize=11)
axes[2].set_title('Experiment C: Dropout', fontsize=12)
axes[2].set_ylim(
    min(r['val_f1'] for r in dropout_results) - 0.02, 1.0)

fig.suptitle('BiLSTM Hyperparameter Experiments',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '17_bilstm_hparam_experiments.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Section 11 -- Summary

In [ ]:
print_section('BILSTM BASELINE SUMMARY')

print('Tokenizer Configuration:')
print(f'  Vocabulary size  : {ACTUAL_VOCAB_SIZE:,}')
print(f'  Max sequence len : {MAX_LEN}')
print(f'  Batch size       : {BATCH_SIZE}')
print()
print('Optimal Hyperparameters (from experiments):')
print(f'  Learning rate    : {OPTIMAL_LR}')
print(f'  LSTM units       : {OPTIMAL_LSTM}')
print(f'  Dropout          : {OPTIMAL_DROPOUT}')
print(f'  Embedding dim    : 128 (fixed)')
print()
print('Final Test Set Results:')
print(f"  Accuracy         : {bilstm_metrics['Accuracy']:.4f}")
print(f"  F1 Macro         : {bilstm_metrics['F1_Macro']:.4f}")
print(f"  ROC-AUC          : {bilstm_metrics['ROC_AUC']:.4f}")
print()
print('Comparison to All Baselines:')
print(f'  Random Forest    : 0.9429 F1')
print(f'  XGBoost          : 0.9627 F1')
print(f'  Logistic Reg.    : 0.9703 F1')
print(f'  LinearSVC        : 0.9720 F1  <- best classical')
print(f"  BiLSTM           : {bilstm_metrics['F1_Macro']:.4f} F1  <- this model")
diff = bilstm_metrics['F1_Macro'] - 0.9720
print(f'  Improvement      : {diff:+.4f}')
print()
print('Files saved to Google Drive:')
print('  models/tokenizer.joblib')
print('  models/bilstm_model.pt')
print('  results/bilstm_results.csv')
print('  figures/results/15_bilstm_training_curves.png')
print('  figures/results/16_bilstm_evaluation.png')
print('  figures/results/17_bilstm_hparam_experiments.png')

## Next Steps

The BiLSTM baseline establishes the neural model performance.
The next notebook builds the 3-branch hybrid model combining:
1. TF-IDF sparse features (from notebook 03)
2. GloVe 100d pretrained embeddings with BiLSTM
3. Handcrafted linguistic features

The hybrid model target is to exceed both the classical
baseline (97.20% F1) and the BiLSTM baseline.